# Unit 10A — Denoiser blocks: timestep embedding and FiLM residual block

**Your result:** the two blocks that tell a denoiser *how noisy its input is* and *what the robot sees*. Your versions will match the pinned LeRobot diffusion-policy blocks, for outputs and gradients, including the weights of the prepared checkpoint.

Two short exercises (about 15 minutes each) and a larger experiment (30–60 minutes). These times are estimates, not measurements. Full answers are in [the reference notebook](10a-denoiser-blocks-reference.ipynb).

**Sources.** [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) (1D temporal U-Net with FiLM conditioning), FiLM: [Perez et al. 2018](https://arxiv.org/abs/1709.07871), and the timestep embedding from [DDPM](https://arxiv.org/abs/2006.11239) (appendix B, after Vaswani et al.). Pinned code: `DiffusionSinusoidalPosEmb`, `DiffusionConv1dBlock` and `DiffusionConditionalResidualBlock1d` in `lerobot/policies/diffusion/modeling_diffusion.py` (revision `e0d50211ef236143ae867228662b7dfaba554f02`).

**Where this unit sits.** In Unit 9 the denoiser was supplied. Here you open it. Unit 10B assembles these blocks into the full U-Net. Your Unit 9 functions are available as package versions; this unit's functions are wired into the real network through hooks.

In [ ]:
# Supplied: workspace, offline caches, imports
from pathlib import Path
import os
import sys


def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import math
import time
import torch
from torch import nn
import matplotlib.pyplot as plt

import vla_lab.teaching.diffusion as ref  # package reference: used by checks only
from vla_lab.teaching.diffusion import (  # Unit 9 functions, as package versions
    ConditionalResidualBlock1d, ConditionalUnet1D, Conv1dBlock, DDPMSchedule, DiffusionDenoiser,
    cosine_betas, epsilon_loss_fn, flat_state_condition, forward_noise,
    load_state_action_windows, take_batch,
)
from vla_lab.teaching.training import train_loop  # from the training unit (package version)
from lerobot.policies.diffusion.modeling_diffusion import (
    DiffusionConditionalResidualBlock1d, DiffusionSinusoidalPosEmb,
)

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual, expected, message, atol=1e-5):
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: how a convolutional denoiser gets its conditions

The denoiser is a 1D convolutional network over the *time axis of the action window* (16 steps). It receives two kinds of side information:

1. **How noisy is the input?** The integer timestep `t` is turned into a smooth vector (`sinusoidal_timestep_embedding`) and passed through a small MLP.
2. **What does the robot see?** The observation features (state, camera features) form one *global condition* vector.

The two are concatenated into a `cond` vector. Every residual block turns `cond` into per-channel numbers that *modulate* the convolution features. This is **FiLM** (feature-wise linear modulation): `features = scale * features + bias`, with `scale` and `bias` computed from `cond`. In the pinned policy `use_film_scale_modulation=True`, so both scale and bias are used.

**Real data used in this unit.** Real LIBERO `bowl_plate` windows (state history of 2 frames and 16 actions with padding mask), read from local files; no download starts here. The recording rate is 10 Hz and is not resampled to the 20 Hz controller. If files are missing, the cell tells you the setup command.

In [ ]:
# Supplied: real windows
TRAIN = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "train")
VALID = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "validation")
print("train windows:", len(TRAIN["action"]), "| validation windows:", len(VALID["action"]))

## Exercise 10A.1 — Sinusoidal timestep embedding (about 10–15 min)

**Purpose.** Give the network a smooth, bounded code for the noise level, instead of a raw integer.

**Contract (shapes, units, masks).**

- `sinusoidal_timestep_embedding(timesteps, dim)`: `timesteps (B,)` integer or float tensor; `dim` even and at least 4. Return `(B, dim)` `float32`.
- `half = dim // 2`. Frequencies: `f_k = exp(-k * log(10000) / (half - 1))` for `k = 0 … half-1`.
- Output row = `[sin(t * f_0) … sin(t * f_{half-1}), cos(t * f_0) … cos(t * f_{half-1})]`: **all sines first, then all cosines** (not interleaved).
- Timestep 0 gives `sin = 0` and `cos = 1` everywhere.

**Method and source.** The Transformer position code applied to the diffusion timestep ([DDPM](https://arxiv.org/abs/2006.11239), appendix B; [Vaswani et al. 2017](https://arxiv.org/abs/1706.03762), 3.5). The layout differs from Exercise 7.4 (that one interleaves sin and cos and counts frequencies differently).

**Pointers.** Make the frequency vector first. Multiply the column of timesteps with the frequency row to get `(B, half)` angles. Concatenate the sines and the cosines on the last axis.

<details><summary>Detailed hint</summary>

```python
half = dim // 2
scale = math.log(10000) / (half - 1)
frequencies = torch.exp(torch.arange(half, device=timesteps.device) * -scale)
angles = timesteps.float().unsqueeze(-1) * frequencies.unsqueeze(0)
return torch.cat((angles.sin(), angles.cos()), dim=-1)
```

</details>

In [ ]:
# Implement: Exercise 10A.1
def sinusoidal_timestep_embedding(timesteps, dim):
    # (B,) -> (B, dim): [sin(t f_0) ... sin(t f_{h-1}), cos(t f_0) ... cos(t f_{h-1})], f_k = exp(-k log(10000)/(h-1)).
    raise NotImplementedError("Complete Exercise 10A.1: sinusoidal_timestep_embedding")

In [ ]:
# Check: Exercise 10A.1
t = torch.tensor([0, 1, 7, 99])
for dim in (16, 32, 128):
    mine = sinusoidal_timestep_embedding(t, dim)
    assert mine.shape == (4, dim) and mine.dtype == torch.float32, f"Expected float32 (4, {dim})."
    close(mine, DiffusionSinusoidalPosEmb(dim)(t), f"dim {dim} differs from the pinned embedding", atol=1e-6)
zero = sinusoidal_timestep_embedding(torch.tensor([0]), 8)
assert torch.equal(zero[0, :4], torch.zeros(4)) and torch.equal(zero[0, 4:], torch.ones(4)), "Timestep 0 is all sines 0 then all cosines 1."
close(sinusoidal_timestep_embedding(torch.tensor([5]), 8)[0, 0], torch.tensor(math.sin(5.0)), "first frequency is 1", atol=1e-6)
close(sinusoidal_timestep_embedding(torch.tensor([5]), 8)[0, 3], torch.tensor(math.sin(5 * 1e-4)), "last frequency is 1/10000", atol=1e-6)
print("Exercise 10A.1 passed")

**Use your function.** Embed all 100 training timesteps. Neighbouring steps should look similar and far steps different.

In [ ]:
# Use your function: Exercise 10A.1
embedding = sinusoidal_timestep_embedding(torch.arange(100), 128)
similarity = embedding @ embedding.T
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].imshow(embedding.T, aspect="auto", cmap="coolwarm"); axes[0].set_xlabel("timestep"); axes[0].set_ylabel("feature")
axes[1].imshow(similarity, cmap="viridis"); axes[1].set_title("similarity between timesteps")
plt.tight_layout(); plt.show()

## Exercise 10A.2 — FiLM-conditioned residual block (about 15–20 min)

**Purpose.** Write the block that is repeated through the whole U-Net: two convolutions, with the condition modulating the features between them.

**Contract (shapes, units, masks).**

- `block` has `conv1`, `conv2` (`Conv1dBlock`: Conv1d → GroupNorm → Mish, keeps the length), `cond_encoder` (`Mish → Linear(cond_dim, K)`), `residual_conv` (`Conv1d(in, out, 1)` or `Identity`), `out_channels` (`C`) and the bool `use_film_scale_modulation`.
- `x (B, C_in, T)` (channels first), `cond (B, cond_dim)`. Return `(B, C, T)`.
- `out = conv1(x)`. `embed = cond_encoder(cond)` reshaped to `(B, K, 1)` so it broadcasts over `T`.
- **With scale modulation** (`K = 2C`): `scale = embed[:, :C]`, `bias = embed[:, C:]`, `out = scale * out + bias`. **Without** (`K = C`): `out = out + embed`.
- `out = conv2(out)`. Return `out + residual_conv(x)`.
- The pinned code uses `scale * out + bias`, not `(1 + scale) * out + bias`.

**Method and source.** FiLM: `γ(c) ⊙ F + β(c)` — [Perez et al. 2018](https://arxiv.org/abs/1709.07871). Diffusion Policy uses it in each 1D residual block ([paper site](https://diffusion-policy.cs.columbia.edu/)). Pinned class: `DiffusionConditionalResidualBlock1d`.

**Pointers.** Follow the order of the contract line by line. The only decision is how to cut `embed` into `scale` and `bias`. The residual connection adds the *input* (after `residual_conv`), not the output of `conv1`.

<details><summary>Detailed hint</summary>

```python
out = block.conv1(x)
embed = block.cond_encoder(cond).unsqueeze(-1)              # (B, K, 1)
if block.use_film_scale_modulation:
    scale, bias = embed[:, : block.out_channels], embed[:, block.out_channels :]
    out = scale * out + bias
else:
    out = out + embed
out = block.conv2(out)
return out + block.residual_conv(x)
```

</details>

In [ ]:
# Implement: Exercise 10A.2
def film_residual_forward(block, x, cond):
    # conv1 -> FiLM modulation from cond -> conv2 -> add residual_conv(x).
    raise NotImplementedError("Complete Exercise 10A.2: film_residual_forward")

In [ ]:
# Check: Exercise 10A.2
for scale in (True, False):
    torch.manual_seed(2)
    oracle = DiffusionConditionalResidualBlock1d(8, 16, 10, kernel_size=5, n_groups=8, use_film_scale_modulation=scale)
    mine = ConditionalResidualBlock1d(8, 16, 10, 5, 8, scale, forward_fn=film_residual_forward)
    mine.load_state_dict(oracle.state_dict(), strict=True)       # same layer names: identical weights
    x, cond = torch.randn(3, 8, 16, requires_grad=True), torch.randn(3, 10, requires_grad=True)
    out = mine(x, cond)
    assert out.shape == (3, 16, 16), "Expected (B, out_channels, T)."
    close(out, oracle(x, cond), f"output (scale modulation={scale}) differs from the pinned block")
    out.square().sum().backward()
    x2, cond2 = (t.detach().clone().requires_grad_(True) for t in (x, cond))
    oracle(x2, cond2).square().sum().backward()
    close(x.grad, x2.grad, f"input gradient (scale={scale})", atol=1e-5)
    close(cond.grad, cond2.grad, f"condition gradient (scale={scale})", atol=1e-5)
    close(mine.cond_encoder[1].weight.grad, oracle.cond_encoder[1].weight.grad, "FiLM layer gradient", atol=1e-5)
    assert not torch.allclose(mine(x, cond + 1.0), out.detach(), atol=1e-4), "The condition must change the output."
same = ConditionalResidualBlock1d(16, 16, 10, 5, 8, True, forward_fn=film_residual_forward)
assert isinstance(same.residual_conv, nn.Identity) and same(torch.randn(2, 16, 16), torch.randn(2, 10)).shape == (2, 16, 16), "Also check equal channels."
print("Exercise 10A.2 passed")

**Use your function.** Condition one block on a real state history and a timestep. Change only the timestep and see how much of the output moves.

In [ ]:
# Use your function: Exercise 10A.2
torch.manual_seed(1)
block = ConditionalResidualBlock1d(7, 16, 32 + 16, 5, 8, True, forward_fn=film_residual_forward).eval()
window = VALID["action"][:4].transpose(1, 2)                             # (4, 7 channels, 16 steps): real action windows
state = VALID["observation.state"][:4].flatten(1)                        # (4, 16): real robot states
def condition(t):
    return torch.cat([sinusoidal_timestep_embedding(torch.full((4,), t), 32), state], dim=-1)
with torch.no_grad():
    base = block(window, condition(10))
    for t in (11, 50, 90):
        print(f"change timestep 10 -> {t}: relative output change", round(float((block(window, condition(t)) - base).norm() / base.norm()), 3))
    other = block(window, torch.cat([sinusoidal_timestep_embedding(torch.full((4,), 10), 32), state.flip(0)], dim=-1))
    print("same timestep, states of other windows: relative change", round(float((other - base).norm() / base.norm()), 3))

### Prepared weights in your block (real-size parity)

The prepared diffusion checkpoint's first U-Net block is copied into a block that runs **your** `film_residual_forward`. The layer names match, so the copy is a plain strict `load_state_dict`. Both blocks receive the same real-sized random input. The check reads the local checkpoint only (no download). If the file is missing, the cell says so.

In [ ]:
RUN_PREPARED_BLOCK_PARITY = True

if RUN_PREPARED_BLOCK_PARITY:
    from safetensors.torch import load_file

    checkpoint = DATA / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
    if not (checkpoint / "model.safetensors").is_file():
        print("Prepared diffusion checkpoint not found; skipping. It is part of the reference bundle (outside this notebook).")
    else:
        tensors = load_file(checkpoint / "model.safetensors")
        prefix = "diffusion.unet.down_modules.0.0."
        state = {key[len(prefix):]: value for key, value in tensors.items() if key.startswith(prefix)}
        channels_in, channels_out = state["conv1.block.0.weight"].shape[1], state["conv1.block.0.weight"].shape[0]
        cond_dim = state["cond_encoder.1.weight"].shape[1]
        prepared = ConditionalResidualBlock1d(channels_in, channels_out, cond_dim, 5, 8, True, forward_fn=film_residual_forward).eval()
        prepared.load_state_dict(state, strict=True)
        oracle = DiffusionConditionalResidualBlock1d(channels_in, channels_out, cond_dim, kernel_size=5, n_groups=8, use_film_scale_modulation=True).eval()
        oracle.load_state_dict(state, strict=True)
        x, cond = torch.randn(4, channels_in, 16), torch.randn(4, cond_dim)
        with torch.no_grad():
            print(f"block {channels_in}->{channels_out} channels, cond {cond_dim}: largest output difference", float((prepared(x, cond) - oracle(x, cond)).abs().max()))
else:
    print("Prepared-block parity is off.")

## Experiment — how should the denoiser use the observation? (30–60 min)

**Question.** Does a small denoiser trained briefly on real windows (a) benefit from FiLM *scale* in addition to *bias*, and (b) use the robot state at all?

**Setup (real data).** A 1D U-Net (`down_dims = (32, 64, 128)`) built from **your** `sinusoidal_timestep_embedding` and `film_residual_forward`, trained with the Unit 9 loss callback on real windows. The condition is the flattened 2×8 state history, or nothing.

**Fixed for every trial.** Cosine schedule, `T = 100`; weight seed `0`; batch seed `1`; **200 updates**, batch 64, Adam(`lr=1e-3`, betas `(0.95, 0.999)`); the same validation windows and noise (seed `7`) for every trial.

**Allowed changes (per trial).** `film_scale` (`True` = scale and bias, `False` = bias only) and `cond` (`"state"` or `"none"`).

**Measures.** `eps_grid`: noise-prediction MSE on validation windows (real steps only), averaged over timesteps `10, 30, 50, 70, 90` with fixed noise. `eps_shuffled`: the same with the states of *other* windows (for `cond="state"` only). `condition_gain = eps_shuffled − eps_grid`: how much the model relies on its own observation. Also the parameter count.

**Prediction first.** In the next cell: will `cond="state"` beat `cond="none"`? Will scale modulation help?

**Keep failed trials.** Every trial is appended to `TRIALS`. No improvement is a valid result.

In [ ]:
# Larger training experiment: run only when you opt in.
RUN_EXPERIMENT = False

In [ ]:
if RUN_EXPERIMENT:
    PREDICTION = "TODO: replace this text with your prediction."
    print(PREDICTION)

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: trial table helper
    import time
    import pandas as pd

    def show_trials(trials, columns):
        """Print every trial that was run, including failed ones, and return a table."""
        table = pd.DataFrame([{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials])
        print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return table

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: harness. Your embedding and your FiLM block are in the real U-Net.
    STEPS, BATCH, LR, SEED_WEIGHTS, SEED_BATCH, SEED_NOISE = 200, 64, 1e-3, 0, 1, 7
    SCHEDULE = DDPMSchedule.cosine(100)
    GRID = (10, 30, 50, 70, 90)
    BASELINE = {"film_scale": True, "cond": "state"}
    TRIALS = []


    def run_trial(config):
        started = time.perf_counter()
        torch.manual_seed(SEED_WEIGHTS)
        use_state = config["cond"] == "state"
        unet = ConditionalUnet1D(
            7, 16 if use_state else 0, (32, 64, 128), step_embed_dim=32,
            use_film_scale_modulation=config["film_scale"],
            block_forward_fn=film_residual_forward, embedding_fn=sinusoidal_timestep_embedding,
        )
        model = DiffusionDenoiser(unet, SCHEDULE, flat_state_condition if use_state else (lambda batch: None))
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.95, 0.999))
        draws = torch.Generator().manual_seed(SEED_BATCH)
        batches = [take_batch(TRAIN, torch.randint(0, len(TRAIN["action"]), (BATCH,), generator=draws)) for _ in range(STEPS)]
        train_loop(model, batches, optimizer, epsilon_loss_fn, steps=STEPS)
        model.eval()
        with torch.no_grad():
            noise = torch.randn(VALID["action"].shape, generator=torch.Generator().manual_seed(SEED_NOISE))
            real = ~VALID["action_is_pad"]
            other = {**VALID, "observation.state": VALID["observation.state"][torch.randperm(len(noise), generator=torch.Generator().manual_seed(3))]}
            own, shuffled = [], []
            for t in GRID:
                times = torch.full((len(noise),), t)
                noisy = forward_noise(VALID["action"], noise, times, SCHEDULE.alpha_bar)
                own.append((model(noisy, times, VALID) - noise).square().mean(-1)[real].mean().item())
                if use_state:
                    shuffled.append((model(noisy, times, other) - noise).square().mean(-1)[real].mean().item())
        result = {"eps_grid": sum(own) / len(own), "parameters": sum(p.numel() for p in model.parameters())}
        result["eps_shuffled"] = sum(shuffled) / len(shuffled) if use_state else float("nan")
        result["condition_gain"] = result["eps_shuffled"] - result["eps_grid"] if use_state else float("nan")
        result["seconds"] = time.perf_counter() - started
        return result


    def record(config):
        config = {**BASELINE, **config}
        try:
            TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
        except Exception as error:  # a failed trial is evidence too
            TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


    record(BASELINE)
    COLUMNS = ["eps_grid", "eps_shuffled", "condition_gain", "parameters", "seconds"]
    show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least two trials that each change one setting. Budget: 200 updates per trial. `condition_gain` is `nan` when there is no condition.

In [ ]:
if RUN_EXPERIMENT:
    # Implement: choose your trials.
    MY_TRIALS = [
        # {"film_scale": False},
    ]
    for trial in MY_TRIALS:
        record(trial)
    show_trials(TRIALS, COLUMNS);

In [ ]:
if RUN_EXPERIMENT:
    # Check: the protocol was followed.
    assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
    assert len(TRIALS) >= 3, "Run the baseline and at least two trials of your own."
    assert all(set(t["config"]) == set(BASELINE) for t in TRIALS), "Only film_scale and cond may change."
    assert all(t["config"]["cond"] in {"state", "none"} for t in TRIALS)
    ok = [t for t in TRIALS if t["status"] == "ok"]
    assert all(torch.isfinite(torch.tensor(t["eps_grid"])) for t in ok), "eps_grid must be finite."
    print(f"{len(TRIALS)} trials kept ({len(ok)} ok). Budget: {STEPS} updates, batch {BATCH}, seeds {SEED_WEIGHTS}/{SEED_BATCH}/{SEED_NOISE}.")

In [ ]:
if RUN_EXPERIMENT:
    # Use your function: plot every trial
    ok = [t for t in TRIALS if t["status"] == "ok"]
    labels = [", ".join(f"{k}={v}" for k, v in t["config"].items() if v != BASELINE[k]) or "baseline" for t in ok]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for axis, column in zip(axes, ["eps_grid", "condition_gain"]):
        axis.bar(range(len(ok)), [t[column] for t in ok])
        axis.set_xticks(range(len(ok))); axis.set_xticklabels(labels, rotation=30, ha="right"); axis.set_title(column)
    plt.tight_layout(); plt.show()

**Explain (2–4 sentences, edit this cell).** Which trial had the lowest `eps_grid`? Was a denoiser with no observation allowed to be better on this measure — and what does that say about judging a policy by noise-prediction loss alone? Does `condition_gain` show that the state is used? What would you add as a condition (Unit 10B and 11 use camera features too)?

*Optional extension:* repeat the baseline with another `SEED_BATCH` to see the spread, and keep all rows.

## What you built

- `sinusoidal_timestep_embedding` and `film_residual_forward`, equal to the pinned blocks in output and gradient, also with the weights of the prepared checkpoint.
- Real-data evidence about conditioning, with every trial kept.

**Next:** Unit 10B stacks these blocks into the full conditional U-Net: temporal down/up-sampling, skip connections, and a check against the pinned network.